# 07 — Matching SST hearings to CAP rows (no topic codes)

**Purpose.** Link each SST hearing (GovInfo `packageId`) to its row in the
CAP congressional-hearings dataset, so that the frozen taxonomy-v0 labels
can be evaluated against CAP's coding in the CAP-evaluation notebook.

**Contamination boundary.** This notebook loads **only matching columns**
(identifiers, chamber, congress, dates, committee codes, titles) via an
explicit `usecols` allowlist — the CAP topic assignment columns
(`majortopic`, `subtopic`, `pap_*`) are never read here. The v0 labels were
hash-frozen before any evaluation contact
(`quality_reports/20260822_taxonomy_v0_freeze.md`).

**Design.** CAP identifies hearings by CIS ids, not GovInfo ids, so the
link is (congress, chamber-committee) blocking + date proximity + title
similarity, with a greedy one-to-one resolution and explicit quality tiers.
CAP ends at Congress 117 (2021), so 2022–2025 SST hearings are unmatchable
by construction and reported as such, not as failures.

**Inputs.** `data/cleaned/01_text_summarization/hearing_summaries.parquet`
(SST metadata); CAP CSV under `data/raw/04_comparative_agenda_project/`.

**Outputs.** `interim/topics/taxonomy_v0/cap_match.parquet` — one row per
matched pair with tier, similarity, and date distance.

## Setup and the two sides

CAP side: rows whose primary or secondary committee is House Science (120)
or Senate Commerce (207), 1996 onward. `latin-1` because the file is not
valid UTF-8. Our side: the 2,352 SST hearings with title, chamber,
congress, and first held date.

In [1]:
import re
import sys

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

from paths import DATA_REPO, ROOT, topics_run_dir

TAX = topics_run_dir("taxonomy_v0")
CAP_CSV = (ROOT / "data" / "raw" / "04_comparative_agenda_project" /
           "US-Legislative_congressional_hearings-21.7.csv")

# Allowlist: matching fields only. majortopic/subtopic/pap_* stay unread.
SAFE_COLS = ["id", "source", "Chamber", "Congress", "year", "Month", "Days",
             "Committee1", "Committee2", "title", "StartDate", "CISYear"]

cap = pd.read_csv(CAP_CSV, encoding="latin-1", usecols=SAFE_COLS,
                  low_memory=False)
for c in ("Committee1", "Committee2"):
    cap[c] = cap[c].astype(str).str.replace(r"\.0$", "", regex=True)
cap = cap.loc[(cap["year"] >= 1996) &
              (cap["Committee1"].isin(["120", "207"]) |
               cap["Committee2"].isin(["120", "207"]))].copy()
cap["committee"] = np.where(cap["Committee1"].isin(["120", "207"]),
                            cap["Committee1"], cap["Committee2"])
cap["date"] = pd.to_datetime(cap["StartDate"], format="%m/%d/%Y",
                             errors="coerce")

ours = pd.read_parquet(
    DATA_REPO / "cleaned" / "01_text_summarization" / "hearing_summaries.parquet",
    columns=["packageId", "title", "congress", "chamber", "year",
             "heldDate_first"])
ours["congress"] = ours["congress"].astype(int)
ours["date"] = pd.to_datetime(ours["heldDate_first"])
# committee side: our HOUSE hearings sit with CAP 120, SENATE with 207;
# JOINT hearings (4) are allowed to match either side.
ours["side"] = ours["chamber"].map({"HOUSE": "120", "SENATE": "207"})

print(f"CAP candidates : {len(cap):,} (dates present {cap['date'].notna().mean():.0%})")
print(f"SST hearings   : {len(ours):,}; matchable window <=117th Congress: "
      f"{(ours['congress'] <= 117).sum():,}")

CAP candidates : 2,426 (dates present 76%)
SST hearings   : 2,352; matchable window <=117th Congress: 2,209


## Title similarity

Both sides' titles are normalized (lowercase, punctuation stripped, boiler
words dropped) to token sets; similarity is the larger of Jaccard and
containment — containment handles the common case where one source
abbreviates the other's long title. Deterministic, no fuzzy libraries.

In [2]:
STOP = {"the", "of", "and", "a", "an", "on", "in", "for", "to", "act",
        "hearing", "hearings", "oversight", "before", "committee", "s",
        "hr", "review", "examining", "fiscal", "year"}

def tokens(title):
    if not isinstance(title, str):
        return frozenset()
    words = re.sub(r"[^a-z0-9 ]", " ", title.lower()).split()
    return frozenset(w for w in words if w not in STOP and len(w) > 1)

def sim(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return max(inter / len(a | b), inter / min(len(a), len(b)))

cap["tok"] = cap["title"].map(tokens)
ours["tok"] = ours["title"].map(tokens)
print("empty token sets — CAP:", (cap['tok'].map(len) == 0).sum(),
      "| ours:", (ours['tok'].map(len) == 0).sum())

empty token sets — CAP: 63 | ours: 1


## Greedy one-to-one matching with tiers

Within each (congress, committee-side) block, every pair is scored; pairs
are accepted best-score-first, each side used at most once. Tiers make the
evidence explicit — a same-day pair needs less title agreement than an
undated one:

- **T1** same date and title sim ≥ 0.5, or sim ≥ 0.85 within one day;
- **T2** within 3 days and sim ≥ 0.35, or CAP date missing and sim ≥ 0.75;
- **T3** remaining best-in-block pairs with sim ≥ 0.55 (date unusable).

Below T3 stays unmatched. Notebook 07 uses T1+T2 as the headline set and
reports T3 sensitivity.

In [3]:
pairs = []
for (cong, side), block in ours.groupby(["congress", "side"], dropna=False):
    cands = cap.loc[(cap["Congress"] == cong)]
    if side is not np.nan and isinstance(side, str):
        cands = cands.loc[cands["committee"] == side]
    for o in block.itertuples():
        for c in cands.itertuples():
            s = sim(o.tok, c.tok)
            if s == 0.0:
                continue
            dd = abs((o.date - c.date).days) if pd.notna(c.date) else None
            if (dd == 0 and s >= 0.5) or (dd is not None and dd <= 1 and s >= 0.85):
                tier = 1
            elif (dd is not None and dd <= 3 and s >= 0.35) or (dd is None and s >= 0.75):
                tier = 2
            elif s >= 0.55:
                tier = 3
            else:
                continue
            pairs.append({"packageId": o.packageId, "cap_id": c.id,
                          "cap_source": c.source, "tier": tier, "sim": s,
                          "date_diff": dd})

pairs = pd.DataFrame(pairs).sort_values(["tier", "sim"],
                                        ascending=[True, False])
used_o, used_c, keep = set(), set(), []
for r in pairs.itertuples():
    if r.packageId in used_o or r.cap_id in used_c:
        continue
    used_o.add(r.packageId); used_c.add(r.cap_id); keep.append(r.Index)
match = pairs.loc[keep].reset_index(drop=True)
match.to_parquet(TAX / "cap_match.parquet", index=False)

in_window = ours.loc[ours["congress"] <= 117, "packageId"]
print(f"matched: {len(match):,} of {len(in_window):,} in-window hearings "
      f"({len(match) / len(in_window):.0%})")
print(match["tier"].value_counts().sort_index().to_string())

matched: 1,796 of 2,209 in-window hearings (81%)
tier
1    1699
2      64
3      33


## Verification

One-to-one by construction is asserted, match rates are broken out by year
(a sharp drop inside the CAP window would flag a blocking bug), and a
random sample of the weakest accepted pairs is printed for eyeball review —
the honesty check on the tier thresholds.

In [4]:
back = pd.read_parquet(TAX / "cap_match.parquet")
assert not back["packageId"].duplicated().any()
assert not back["cap_id"].duplicated().any()

m = ours.merge(back, on="packageId", how="left")
rate = (m.loc[m["congress"] <= 117].groupby("year")["cap_id"]
          .agg(lambda s: s.notna().mean()))
print("1) match rate by year (in CAP window):")
print("   " + " ".join(f"{y}:{r:.0%}" for y, r in rate.items()))
print(f"2) overall in-window rate: "
      f"{m.loc[m['congress'] <= 117, 'cap_id'].notna().mean():.0%} "
      f"| tiers: {back['tier'].value_counts().sort_index().to_dict()}")
print(f"3) out-of-window (118th+): {(m['congress'] > 117).sum():,} hearings, "
      f"unmatchable by construction")

weak = back.sort_values("sim").head(8).merge(
    ours[["packageId", "title", "date"]], on="packageId").merge(
    cap[["id", "title", "date"]].rename(
        columns={"id": "cap_id", "title": "cap_title", "date": "cap_date"}),
    on="cap_id")
print("\n4) weakest accepted pairs (eyeball):")
for r in weak.itertuples():
    print(f"   T{r.tier} sim={r.sim:.2f} dd={r.date_diff}")
    print(f"     SST: {str(r.title)[:90]}")
    print(f"     CAP: {str(r.cap_title)[:90]}")

1) match rate by year (in CAP window):
   1996:100% 1997:100% 1999:98% 2000:100% 2001:97% 2002:100% 2003:98% 2004:90% 2005:100% 2006:100% 2007:99% 2008:99% 2009:87% 2010:98% 2011:89% 2012:100% 2013:100% 2014:98% 2015:97% 2016:87% 2017:73% 2018:25% 2019:22% 2020:37% 2021:18% 2022:2%
2) overall in-window rate: 81% | tiers: {1: 1699, 2: 64, 3: 33}
3) out-of-window (118th+): 143 hearings, unmatchable by construction

4) weakest accepted pairs (eyeball):
   T3 sim=0.57 dd=nan
     SST: FOSSIL ENERGY RESEARCH: ENABLING OUR CLEAN ENERGY FUTURE
     CAP: From Lab to Market: Accelerating Our Progress Toward Economic Recovery and a Clean Energy 
   T3 sim=0.60 dd=nan
     SST: SCIENCE AND ENERGY RESEARCH INFRASTRUCTURE NEEDS OF THE U.S. DEPARTMENT OF ENERGY
     CAP: Department of Energy Science for the Future ActDepartment of Energy Science for the Future
   T1 sim=0.67 dd=0.0
     SST: NASA FY 2008 BUDGET
     CAP: NASA FY2008 Budget
   T3 sim=0.67 dd=5179.0
     SST: PIPELINE SAFETY IN THE GR